# Gradientes y jacobianos: cálculo a mano y verificación con Python

En cada ejercicio seguimos el mismo esquema:

1. **Derivamos a mano** el gradiente o el jacobiano.
2. Lo **programamos** con NumPy.
3. Lo **verificamos** de dos formas independientes:
   - **simbólicamente** con SymPy (cuando la función tiene pocas variables), y
   - **numéricamente** con diferencias finitas centradas (`gradcheck.py`).

**Convenciones.** Para $f:\mathbb{R}^n\to\mathbb{R}$, el gradiente es el vector
$\nabla f(x) = \left(\tfrac{\partial f}{\partial x_1},\dots,\tfrac{\partial f}{\partial x_n}\right)$.
Para $F:\mathbb{R}^n\to\mathbb{R}^m$, el jacobiano es la matriz $m\times n$ con
$J_{ij} = \tfrac{\partial F_i}{\partial x_j}$ (una fila por componente de salida).

In [1]:
import numpy as np
import sympy as sp

from gradcheck import (
    error_relativo,
    gradiente_numerico,
    jacobiano_numerico,
    verificar_gradiente,
    verificar_jacobiano,
)

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)

## 0. ¿Cómo funciona la verificación numérica?

La diferencia centrada

$$\frac{\partial f}{\partial x_i}(x) \approx \frac{f(x + h\,e_i) - f(x - h\,e_i)}{2h}$$

tiene error de truncamiento $O(h^2)$, pero al restar dos números casi iguales aparece
un error de redondeo $O(\varepsilon/h)$. Por eso existe un $h$ óptimo (en torno a
$10^{-5}$–$10^{-6}$ en doble precisión): ni demasiado grande ni demasiado pequeño.
Lo vemos con $f(x)=e^x$ en $x=1$, cuya derivada exacta es $e$.

In [2]:
print(f"{'h':>8}  {'adelante':>10}  {'centrada':>10}")
for h in [1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    adelante = (np.exp(1 + h) - np.exp(1)) / h
    centrada = (np.exp(1 + h) - np.exp(1 - h)) / (2 * h)
    print(f"{h:8.0e}  {abs(adelante - np.e):10.2e}  {abs(centrada - np.e):10.2e}")

       h    adelante    centrada
   1e-01    1.41e-01    4.53e-03
   1e-02    1.36e-02    4.53e-05
   1e-04    1.36e-04    4.53e-09
   1e-06    1.36e-06    1.63e-10
   1e-08    6.60e-09    6.60e-09
   1e-10    1.55e-06    6.73e-07
   1e-12    4.32e-04    2.10e-04


## 1. Gradiente de una función escalar de dos variables

$$f(x, y) = x^2 y + \sin(xy)$$

Derivando respecto a cada variable (regla de la cadena en el seno):

$$\nabla f(x,y) = \begin{pmatrix} 2xy + y\cos(xy) \\ x^2 + x\cos(xy) \end{pmatrix}$$

In [3]:
def f1(v):
    x, y = v
    return x**2 * y + np.sin(x * y)

def grad_f1(v):
    x, y = v
    return np.array([2 * x * y + y * np.cos(x * y), x**2 + x * np.cos(x * y)])

# Verificación simbólica
x, y = sp.symbols("x y")
expr = x**2 * y + sp.sin(x * y)
grad_sym = [sp.diff(expr, s) for s in (x, y)]
display(sp.Matrix(grad_sym))
a_mano = [2 * x * y + y * sp.cos(x * y), x**2 + x * sp.cos(x * y)]
assert all(sp.simplify(g - m) == 0 for g, m in zip(grad_sym, a_mano))
print("SymPy coincide con la derivación a mano")

# Verificación numérica
p = np.array([1.3, -0.7])
print("analítico:", grad_f1(p))
print("numérico: ", gradiente_numerico(f1, p))
assert verificar_gradiente(f1, grad_f1, p, nombre="ejercicio 1")

Matrix([
[2*x*y + y*cos(x*y)],
[ x**2 + x*cos(x*y)]])

SymPy coincide con la derivación a mano
analítico: [-2.249622  2.487869]
numérico:  [-2.249622  2.487869]
[OK   ] ejercicio 1: error relativo = 3.90e-11 (tol = 1e-06)


## 2. Forma cuadrática

$$f(x) = x^\top A x + b^\top x, \qquad A\in\mathbb{R}^{n\times n},\; b\in\mathbb{R}^n$$

En componentes, $f(x)=\sum_{i,j} A_{ij}x_i x_j + \sum_i b_i x_i$, así que

$$\frac{\partial f}{\partial x_k} = \sum_j A_{kj}x_j + \sum_i A_{ik}x_i + b_k
\quad\Longrightarrow\quad \nabla f(x) = (A + A^\top)\,x + b$$

Un error muy común es escribir $2Ax + b$, que **solo es correcto si $A$ es simétrica**.
La verificación numérica lo detecta.

In [4]:
n = 5
A = rng.normal(size=(n, n))   # no simétrica a propósito
b = rng.normal(size=n)
x0 = rng.normal(size=n)

f2 = lambda x: x @ A @ x + b @ x
grad_f2 = lambda x: (A + A.T) @ x + b
grad_f2_mal = lambda x: 2 * A @ x + b

assert verificar_gradiente(f2, grad_f2, x0, nombre="(A + A^T) x + b")
assert not verificar_gradiente(f2, grad_f2_mal, x0, nombre="2 A x + b (incorrecto)")

[OK   ] (A + A^T) x + b: error relativo = 8.45e-11 (tol = 1e-06)
[FALLA] 2 A x + b (incorrecto): error relativo = 5.63e-01 (tol = 1e-06)


## 3. Mínimos cuadrados

$$f(x) = \lVert Ax - b\rVert^2, \qquad A\in\mathbb{R}^{m\times n}$$

Con $r = Ax - b$ tenemos $f = r^\top r$, y por la regla de la cadena
$\nabla f = J_r^\top\,(2r)$ con $J_r = A$:

$$\nabla f(x) = 2A^\top(Ax - b)$$

Además, el gradiente debe anularse en la solución de mínimos cuadrados.

In [5]:
m, n = 8, 4
A = rng.normal(size=(m, n))
b = rng.normal(size=m)
x0 = rng.normal(size=n)

f3 = lambda x: np.sum((A @ x - b) ** 2)
grad_f3 = lambda x: 2 * A.T @ (A @ x - b)

assert verificar_gradiente(f3, grad_f3, x0, nombre="ejercicio 3")

x_opt = np.linalg.lstsq(A, b, rcond=None)[0]
print("||grad|| en el óptimo:", np.linalg.norm(grad_f3(x_opt)))
assert np.linalg.norm(grad_f3(x_opt)) < 1e-10

[OK   ] ejercicio 3: error relativo = 1.04e-10 (tol = 1e-06)
||grad|| en el óptimo: 1.7271017045521525e-15


## 4. Log-sum-exp

$$f(z) = \log\sum_{k=1}^{n} e^{z_k}
\quad\Longrightarrow\quad
\frac{\partial f}{\partial z_i} = \frac{e^{z_i}}{\sum_k e^{z_k}} = \operatorname{softmax}(z)_i$$

Es decir, $\nabla f(z) = \operatorname{softmax}(z)$: sus componentes son positivas y suman 1.

In [6]:
def softmax(z):
    e = np.exp(z - np.max(z))   # restar el máximo evita desbordamientos
    return e / e.sum()

def logsumexp(z):
    c = np.max(z)
    return c + np.log(np.sum(np.exp(z - c)))

z0 = rng.normal(size=6)
assert verificar_gradiente(logsumexp, softmax, z0, nombre="ejercicio 4")
print("suma del gradiente:", softmax(z0).sum())

[OK   ] ejercicio 4: error relativo = 9.15e-11 (tol = 1e-06)
suma del gradiente: 1.0


## 5. Jacobiano del cambio a coordenadas polares

$$F(r,\theta) = \begin{pmatrix} r\cos\theta \\ r\sin\theta \end{pmatrix}
\quad\Longrightarrow\quad
J_F(r,\theta) = \begin{pmatrix} \cos\theta & -r\sin\theta \\ \sin\theta & \phantom{-}r\cos\theta \end{pmatrix},
\qquad \det J_F = r$$

El determinante $r$ es el factor que aparece en $dx\,dy = r\,dr\,d\theta$.

In [7]:
def polares(p):
    r, th = p
    return np.array([r * np.cos(th), r * np.sin(th)])

def jac_polares(p):
    r, th = p
    return np.array([[np.cos(th), -r * np.sin(th)],
                     [np.sin(th),  r * np.cos(th)]])

# Simbólico
r, th = sp.symbols("r theta", positive=True)
F_sym = sp.Matrix([r * sp.cos(th), r * sp.sin(th)])
J_sym = F_sym.jacobian([r, th])
display(J_sym)
assert sp.simplify(J_sym.det()) == r
print("det J =", sp.simplify(J_sym.det()))

# Numérico
p = np.array([2.0, 0.6])
assert verificar_jacobiano(polares, jac_polares, p, nombre="ejercicio 5")
assert abs(np.linalg.det(jacobiano_numerico(polares, p)) - p[0]) < 1e-6

Matrix([
[cos(theta), -r*sin(theta)],
[sin(theta),  r*cos(theta)]])

det J = r
[OK   ] ejercicio 5: error relativo = 5.93e-11 (tol = 1e-06)


## 6. Jacobiano de una función $\mathbb{R}^3\to\mathbb{R}^2$

$$F(x,y,z) = \begin{pmatrix} x\,y\,z \\ x^2 + e^{y}\sin z \end{pmatrix}
\quad\Longrightarrow\quad
J_F = \begin{pmatrix} yz & xz & xy \\ 2x & e^{y}\sin z & e^{y}\cos z \end{pmatrix}$$

El jacobiano no es cuadrado: 2 filas (salidas) por 3 columnas (entradas).
Cada fila es el gradiente de una componente de $F$.

In [8]:
def F6(v):
    x, y, z = v
    return np.array([x * y * z, x**2 + np.exp(y) * np.sin(z)])

def jac_F6(v):
    x, y, z = v
    return np.array([[y * z, x * z, x * y],
                     [2 * x, np.exp(y) * np.sin(z), np.exp(y) * np.cos(z)]])

# Simbólico: evaluamos el jacobiano de SymPy en un punto y comparamos
x, y, z = sp.symbols("x y z")
J_sym = sp.Matrix([x * y * z, x**2 + sp.exp(y) * sp.sin(z)]).jacobian([x, y, z])
display(J_sym)
J_fun = sp.lambdify((x, y, z), J_sym, "numpy")

v0 = np.array([0.5, -1.2, 2.0])
assert error_relativo(np.array(J_fun(*v0), dtype=float), jac_F6(v0)) < 1e-12
print("SymPy coincide con la derivación a mano; forma:", jac_F6(v0).shape)

assert verificar_jacobiano(F6, jac_F6, v0, nombre="ejercicio 6")

Matrix([
[y*z,           x*z,           x*y],
[2*x, exp(y)*sin(z), exp(y)*cos(z)]])

SymPy coincide con la derivación a mano; forma: (2, 3)
[OK   ] ejercicio 6: error relativo = 3.43e-11 (tol = 1e-06)


## 7. Jacobiano de softmax

$$s_i(z) = \frac{e^{z_i}}{\sum_k e^{z_k}}
\quad\Longrightarrow\quad
\frac{\partial s_i}{\partial z_j} = s_i(\delta_{ij} - s_j)
\quad\Longrightarrow\quad
J_s(z) = \operatorname{diag}(s) - s\,s^\top$$

Propiedades que también comprobamos: $J_s$ es simétrica y sus filas suman 0
(porque $\sum_i s_i = 1$ es constante).

In [9]:
jac_softmax = lambda z: np.diag(softmax(z)) - np.outer(softmax(z), softmax(z))

z0 = rng.normal(size=5)
J = jac_softmax(z0)
assert verificar_jacobiano(softmax, jac_softmax, z0, nombre="ejercicio 7")
assert np.allclose(J, J.T)
assert np.allclose(J.sum(axis=1), 0)
print("simétrica y con filas de suma cero")

[OK   ] ejercicio 7: error relativo = 3.28e-11 (tol = 1e-06)
simétrica y con filas de suma cero


## 8. Regla de la cadena con jacobianos

Si $h = G\circ F$ con $F:\mathbb{R}^n\to\mathbb{R}^m$ y $G:\mathbb{R}^m\to\mathbb{R}^p$:

$$J_h(x) = J_G\big(F(x)\big)\; J_F(x)$$

Tomamos $F(x) = \tanh(Wx + c)$ (una capa de red neuronal) y $G = \operatorname{softmax}$:

$$J_F(x) = \operatorname{diag}\!\big(1 - \tanh^2(Wx + c)\big)\,W$$

In [10]:
n, m = 4, 3
W = rng.normal(size=(m, n))
c = rng.normal(size=m)
x0 = rng.normal(size=n)

F8 = lambda x: np.tanh(W @ x + c)
jac_F8 = lambda x: np.diag(1 - np.tanh(W @ x + c) ** 2) @ W

h8 = lambda x: softmax(F8(x))
jac_h8 = lambda x: jac_softmax(F8(x)) @ jac_F8(x)   # regla de la cadena

assert verificar_jacobiano(F8, jac_F8, x0, nombre="capa tanh")
assert verificar_jacobiano(h8, jac_h8, x0, nombre="composición (cadena)")
print("forma de J_h:", jac_h8(x0).shape)

[OK   ] capa tanh: error relativo = 6.06e-11 (tol = 1e-06)
[OK   ] composición (cadena): error relativo = 6.09e-11 (tol = 1e-06)
forma de J_h: (3, 4)


## 9. Gradiente de la regresión logística

Con datos $X\in\mathbb{R}^{N\times d}$, etiquetas $y\in\{0,1\}^N$ y
$\sigma(t) = 1/(1+e^{-t})$, la entropía cruzada media es

$$L(w) = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i\log\sigma(x_i^\top w) + (1-y_i)\log\big(1-\sigma(x_i^\top w)\big)\Big]$$

Usando $\sigma' = \sigma(1-\sigma)$ los términos se simplifican y queda

$$\nabla L(w) = \frac{1}{N}\,X^\top\big(\sigma(Xw) - y\big)$$

In [11]:
N, d = 50, 4
X = rng.normal(size=(N, d))
y = (rng.random(N) < 0.5).astype(float)
w0 = rng.normal(size=d)

sigma = lambda t: 1 / (1 + np.exp(-t))

def perdida(w):
    t = X @ w
    # log(1 + e^t) - y t  es la entropía cruzada escrita de forma estable
    return np.mean(np.logaddexp(0, t) - y * t)

grad_perdida = lambda w: X.T @ (sigma(X @ w) - y) / N

assert verificar_gradiente(perdida, grad_perdida, w0, nombre="ejercicio 9")

[OK   ] ejercicio 9: error relativo = 1.05e-10 (tol = 1e-06)


## 10. Gradiente respecto a una matriz

$$f(W) = \tfrac12\lVert WX - Y\rVert_F^2
\quad\Longrightarrow\quad
\nabla_W f = (WX - Y)\,X^\top$$

El gradiente tiene la misma forma que $W$. `gradiente_numerico` perturba cada
entrada de la matriz por separado, así que sirve igual.

In [12]:
W0 = rng.normal(size=(3, 4))
Xm = rng.normal(size=(4, 6))
Ym = rng.normal(size=(3, 6))

f10 = lambda W: 0.5 * np.sum((W @ Xm - Ym) ** 2)
grad_f10 = lambda W: (W @ Xm - Ym) @ Xm.T

assert grad_f10(W0).shape == W0.shape
assert verificar_gradiente(f10, grad_f10, W0, nombre="ejercicio 10")

[OK   ] ejercicio 10: error relativo = 2.83e-10 (tol = 1e-06)


## Ejercicios propuestos

Deriva a mano, programa y verifica con `verificar_gradiente` / `verificar_jacobiano`:

1. $f(x) = \lVert x\rVert_2$ para $x\neq 0$. (Sol.: $x/\lVert x\rVert$)
2. $f(x) = \log\big(1 + e^{a^\top x}\big)$. (Sol.: $\sigma(a^\top x)\,a$)
3. Coordenadas esféricas $F(\rho,\theta,\varphi) = (\rho\sin\varphi\cos\theta,\ \rho\sin\varphi\sin\theta,\ \rho\cos\varphi)$.
   Comprueba que $|\det J_F| = \rho^2\sin\varphi$.
4. $F(x) = x/\lVert x\rVert$. (Sol.: $J = \big(I - \hat x\hat x^\top\big)/\lVert x\rVert$ con $\hat x = x/\lVert x\rVert$)
5. $f(x) = \tfrac12\,x^\top A x$ con $A$ simétrica: verifica además que el jacobiano del
   gradiente (la **hessiana**) es $A$, usando `jacobiano_numerico` sobre $\nabla f$.